# KS_EBW_Boosted_code

Converted from KS_EBW_Boosted_code.py into a notebook. This notebook keeps the original functions and workflow but adapts a few items for interactive use (for example, OUT_DIR is set to the notebook working directory).

In [1]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.tree import DecisionTreeClassifier

# Download entropy_common.py if not already present
if not os.path.exists('entropy_common.py'):
    !wget https://raw.githubusercontent.com/servantjoseph/Entropy_Balancing/main/entropy_common.py

from entropy_common import (
    normalize_weights, effective_sample_size, sigmoid, eb_fit, eb_weights,
    pairwise_products, compact_leaf_ids_for_two, props, fit_balance_tree, hybrid
)

# Use current working directory as OUT_DIR when running interactively
OUT_DIR = os.getcwd()
SEED = 20260625

--2026-09-24 17:32:53--  https://raw.githubusercontent.com/servantjoseph/Entropy_Balancing/main/entropy_common.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5846 (5.7K) [text/plain]
Saving to: ‘entropy_common.py’

entropy_common.py   100%[===================>]   5.71K  --.-KB/s    in 0s      

2026-09-24 17:32:53 (46.9 MB/s) - ‘entropy_common.py’ saved [5846/5846]



In [2]:
def generate_ks(n=1000, rng=None):
    """Generate Kang--Schafer-style data: Z (latent), X (transformed), y (response), r (response indicator)."""
    if rng is None:
        rng = np.random.default_rng()
    Z = rng.normal(size=(n, 4)); z1, z2, z3, z4 = Z.T
    X = np.column_stack([
        np.exp(z1 / 2.0),
        z2 / (1 + np.exp(z1)) + 10.0,
        (z1 * z3 / 25.0 + 0.6) ** 3,
        (z2 + z4 + 20.0) ** 2,
    ])
    y = 210 + 27.4 * z1 + 13.7 * z2 + 13.7 * z3 + 13.7 * z4 + rng.normal(scale=1.0, size=n)
    p = sigmoid(-z1 + 0.5 * z2 - 0.25 * z3 - 0.1 * z4)
    r = rng.binomial(1, p, size=n).astype(bool)
    return Z, X, y, r


In [3]:
def one_rep(rng, n=1000, B=100, nu=0.10):
    Z, X, y, resp = generate_ks(n, rng)
    mu = X.mean(0)
    sd = X.std(0)
    sd[sd < 1e-12] = 1
    Xs_all = (X - mu) / sd
    Xt = Xs_all
    Xr = Xs_all[resp]
    yr = y[resp]
    Zr = Z[resp]
    if Xr.shape[0] < 50:
        raise RuntimeError('too few respondents')
    target = float(y.mean())
    mu_main = Xt.mean(0)
    Pr = pairwise_products(Xr)
    Pt = pairwise_products(Xt)
    Xpair = np.hstack([Xr, Pr])
    mu_pair = np.r_[mu_main, Pt.mean(0)]
    q = np.ones(Xr.shape[0]) / Xr.shape[0]
    out = []
    def add(method, est, w=None, trees=np.nan):
        row = {
            'method': method,
            'estimate': est,
            'error': est - target,
            'ess': np.nan,
            'main_l2': np.nan,
            'pair_l2': np.nan,
            'latent_z_l2': np.nan,
            'num_trees': trees,
        }
        if w is not None:
            row['ess'] = effective_sample_size(w)
            row['main_l2'] = float(np.linalg.norm(Xr.T @ w - mu_main))
            row['pair_l2'] = float(np.linalg.norm(Pr.T @ w - Pt.mean(0)))
            # latent Z discrepancy: compare weighted respondent Z moments to full-sample Z mean
            try:
                row['latent_z_l2'] = float(np.linalg.norm(Zr.T @ w - Z.mean(0)))
            except Exception:
                row['latent_z_l2'] = np.nan
        out.append(row)
    # Naive respondents
    q0 = np.ones(Xr.shape[0]) / Xr.shape[0]
    add('Naive respondents', float(np.mean(yr)), q0, 0)
    # OLS prediction based on respondents
    A = np.column_stack([np.ones(Xr.shape[0]), Xr])
    beta = np.linalg.lstsq(A, yr, rcond=None)[0]
    pred = np.column_stack([np.ones(n), Xt]) @ beta
    add('OLS prediction', float(np.mean(pred)), None, np.nan)
    # EB on main moments
    w_main, lam_main = eb_fit(Xr, mu_main, q=q, max_iter=80, tol=1e-8)
    add('EB main', float(w_main @ yr), w_main, 0)
    # EB on pairwise moments
    w_pair, lam_pair = eb_fit(Xpair, mu_pair, q=q, max_iter=100, tol=1e-8)
    add('EB pairwise', float(w_pair @ yr), w_pair, 0)
    rs = int(rng.integers(0, 2 ** 31 - 1))
    # EB-offset main hybrid
    w_hyb, nt = hybrid(Xr, Xt, Xr, mu_main, q0=w_main, B=B, nu=nu, random_state=rs)
    add('EB-offset hybrid: main', float(w_hyb @ yr), w_hyb, nt)
    # EB-offset pairwise hybrid
    w_hyb_p, ntp = hybrid(Xr, Xt, Xpair, mu_pair, q0=w_pair, B=B, nu=nu, random_state=rs + 10000)
    add('EB-offset hybrid: pairwise', float(w_hyb_p @ yr), w_hyb_p, ntp)
    return out


In [4]:
def one_rep_seed(args):
    seed, n, B, nu = args
    rng = np.random.default_rng(seed)
    return one_rep(rng, n=n, B=B, nu=nu)


In [5]:
def run(R=1000, n=1000, B=100, nu=0.10, n_jobs=None):
    import multiprocessing as mp
    seeds = [SEED + r for r in range(R)]
    args = [(s, n, B, nu) for s in seeds]
    if n_jobs is None:
        n_jobs = min(8, max(1, (os.cpu_count() or 2) - 1))
    rows = []
    if n_jobs <= 1:
        for r, arg in enumerate(args):
            for row in one_rep_seed(arg):
                row['rep'] = r
                rows.append(row)
    else:
        with mp.Pool(processes=n_jobs) as pool:
            for r, rep_rows in enumerate(pool.imap(one_rep_seed, args, chunksize=5)):
                for row in rep_rows:
                    row['rep'] = r
                    rows.append(row)
    raw = pd.DataFrame(rows)
    summ = []
    for method, g in raw.groupby('method'):
        e = g['error'].to_numpy()
        summ.append({
            'method': method,
            'bias': e.mean(),
            'rmse': math.sqrt(np.mean(e ** 2)),
            'mae': np.mean(np.abs(e)),
            'ess_mean': g['ess'].mean(),
            'main_l2_mean': g['main_l2'].mean(),
            'pair_l2_mean': g['pair_l2'].mean(),
            'latent_z_l2_mean': g['latent_z_l2'].mean(),
            'num_trees_mean': g['num_trees'].mean(),
        })
    return raw, pd.DataFrame(summ)


In [8]:
def main_and_save(R=1000, n=1000, B=100, nu=0.10, n_jobs=4, out_dir=OUT_DIR):
    raw, summary = run(R=R, n=n, B=B, nu=nu, n_jobs=n_jobs)
    raw.to_csv(os.path.join(out_dir, 'ks_sim_raw_rows.csv'), index=False)
    summary.to_csv(os.path.join(out_dir, 'ks_sim_summary.csv'), index=False)
    order = [
        'Naive respondents',
        'OLS prediction',
        'EB main',
        'EB pairwise',
        'EB-offset hybrid: main',
        'EB-offset hybrid: pairwise',
    ]
    labels = ['Naive', 'OLS', 'EB main', 'EB pairwise', 'Hybrid EB-O', 'Hybrid EB-O(p)']
    sm = summary.set_index('method').loc[[m for m in order if m in summary['method'].values]].reset_index()
    plt.figure(figsize=(7.8, 4.2))
    plt.bar(labels[:len(sm)], sm['rmse'])
    plt.ylabel('RMSE')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()
    # latent z L2 plot if available
    z = sm[sm['latent_z_l2_mean'].notna()]
    if not z.empty:
        plt.figure(figsize=(7.8, 4.2))
        plt.bar([labels[order.index(m)] for m in z['method']], z['latent_z_l2_mean'])
        plt.ylabel('Mean latent Z L2')
        plt.xticks(rotation=25, ha='right')
        plt.tight_layout()
        plt.show()
    print(sm.to_string(index=False))
    return raw, summary


def main():
    raw,summary=run(R=1000,n=1000,B=100,nu=0.10,n_jobs=8)
    raw.to_csv(    os.path.join(OUT_DIR,'ks_sim_raw_rows.csv'),index=False);
    summary.to_csv(os.path.join(OUT_DIR,'ks_sim_summary.csv' ),index=False)
    order=['Naive respondents','OLS prediction','EB main','EB pairwise','EB-offset hybrid: main','EB-offset hybrid: pairwise'];
    labels=['Naive','OLS','EB main','EB pairwise','Hybrid EB-O','Hybrid EB-O (pair)']
    sm=summary.set_index('method').loc[order].reset_index()

    plt.figure(figsize=(7.8,4.2)); plt.bar(labels,sm['rmse']); plt.ylabel('RMSE'); plt.xticks(rotation=25,ha='right'); plt.tight_layout();
    plt.savefig(os.path.join(OUT_DIR,'ks_fig_rmse_R1000.pdf'));
    plt.close()

    z=sm[sm['latent_z_l2_mean'].notna()]
    plt.figure(figsize=(7.8,4.2)); plt.bar([labels[order.index(m)] for m in z['method']],z['latent_z_l2_mean']); plt.ylabel('Mean latent Z L2'); plt.xticks(rotation=25,ha='right'); plt.tight_layout();
    plt.savefig(os.path.join(OUT_DIR,'ks_fig_z_l2_R1000.pdf'));
    plt.close()
    print(sm.to_string(index=False)); plt.close('all')

main() #if __name__=='__main__': main()



                    method       bias      rmse       mae   ess_mean  main_l2_mean  pair_l2_mean  latent_z_l2_mean  num_trees_mean
         Naive respondents -10.038451 10.104468 10.038451 499.189000  4.276140e-01  1.834691e-01          0.458066           0.000
            OLS prediction  -0.804651  1.279062  1.063510        NaN           NaN           NaN               NaN             NaN
                   EB main  -1.968277  2.214460  1.984731 368.710299  9.843744e-10  2.671059e-01          0.165067           0.000
               EB pairwise  -0.680973  1.332588  1.077856 332.990381  3.090212e-10  4.382765e-10          0.077834           0.000
    EB-offset hybrid: main  -1.479282  1.686718  1.503299 363.531207  1.254313e-09  2.245975e-01          0.143192           8.650
EB-offset hybrid: pairwise  -0.378916  1.121719  0.902938 329.393114  5.290457e-10  6.996838e-10          0.072732          13.402


## Save Results to GitHub

To save the generated files to your GitHub repository, we need to perform the following steps:

1.  **Generate a GitHub Personal Access Token (PAT)**: This token will allow Colab to interact with your GitHub repository. Follow these steps:
    *   Go to your GitHub profile settings.
    *   Navigate to `Developer settings` -> `Personal access tokens` -> `Tokens (classic)`.
    *   Click `Generate new token` and then `Generate new token (classic)`.
    *   Give it a descriptive name (e.g., `colab_repo_access`).
    *   Set an expiration (e.g., 30 days or 90 days).
    *   Under `Repository permissions`, grant `repo` scope (full control of private repositories).
    *   Click `Generate token` and **copy the token immediately**.

2.  **Store the PAT in Colab Secrets**: Colab provides a secure way to store sensitive information like API keys and tokens.
    *   In the left sidebar of Colab, click the 🔑 icon (Secrets).
    *   Click `Add new secret`.
    *   For the name, enter `GITHUB_TOKEN`.
    *   For the value, paste your GitHub PAT.
    *   Make sure `Notebook access` is toggled ON for this notebook.

Once you have set up the `GITHUB_TOKEN` secret, run the following code cells.

In [ ]:
# Import necessary libraries
import os
from google.colab import userdata

# Retrieve the GitHub token from Colab secrets
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

# Define your GitHub repository details
REPO_OWNER = 'servantjoseph'
REPO_NAME = 'Entropy_Balancing'
BRANCH = 'main'
REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

# Construct the authenticated URL for cloning
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

# Define the target folder within the repository
TARGET_FOLDER = 'OutputData'

# Clone the repository (if not already cloned)
repo_dir = REPO_NAME
if not os.path.exists(repo_dir):
    print(f'Cloning {REPO_NAME}...')
    !git clone {AUTH_REPO_URL}
    print('Repository cloned successfully.')
else:
    print(f'Repository {REPO_NAME} already cloned. Pulling latest changes...')
    %cd {repo_dir}
    !git pull origin {BRANCH}
    %cd ..

# Change to the repository directory to work within it
%cd {repo_dir}

# Create the target folder if it doesn't exist
output_path = os.path.join(repo_dir, TARGET_FOLDER)
os.makedirs(output_path, exist_ok=True)

print(f'Current working directory: {os.getcwd()}')
print(f'Output files will be saved to: {output_path}')

In [ ]:
# List of files to copy
files_to_copy = [
    'ks_sim_raw_rows.csv',
    'ks_sim_summary.csv',
    'ks_fig_rmse_R1000.pdf',
    'ks_fig_z_l2_R1000.pdf'
]

# Copy files from current working directory (OUT_DIR) to the target folder
for filename in files_to_copy:
    source_path = os.path.join('/content', filename) # Assuming OUT_DIR is /content
    destination_path = os.path.join(TARGET_FOLDER, filename)
    if os.path.exists(source_path):
        !cp "{source_path}" "{destination_path}"
        print(f'Copied {filename} to {destination_path}')
    else:
        print(f'Warning: {filename} not found in {source_path}. Skipping.')


In [ ]:
# Configure Git and commit changes
!git config user.email "colab-user@example.com" # Replace with your GitHub email if you want specific attribution
!git config user.name "Colab User" # Replace with your GitHub username

!git add {TARGET_FOLDER}/*
!git commit -m "Update simulation results and figures from Colab" || echo "No changes to commit."

# Push changes to the repository
print(f'Pushing changes to {BRANCH} branch...')
!git push {AUTH_REPO_URL} {BRANCH}
print('Files successfully pushed to GitHub!')

# Navigate back to the original working directory
%cd /content